# Accent Neutralizer for Speech

A Deep Learning + Speech Processing project that transforms speech from non-native accents toward a native-English reference accent, while aiming to preserve the original sentence content.

**GSSoC 2026 Contribution — Issue #1956**

## Overview

This v1 implementation focuses on **acoustic feature conversion** (MFCC-level) as a baseline approach. Given a speech clip in a non-native accent (e.g., Hindi-accented or Chinese-accented English), the model learns to map its MFCC features toward the corresponding features of native-accent speech, using a BiLSTM-based sequence model trained on parallel accented/native sentence pairs.

**Scope note:** true accent conversion is an open research problem — accent and speaker identity are entangled in the same acoustic features (MFCCs, formants), so a model attempting to change *only* accent while perfectly preserving voice identity has to disentangle signals that overlap heavily. This v1 targets a scoped baseline (2 accent pairs) rather than a general-purpose, production-grade solution. "Neutral" here refers to a specific reference accent (General American, via the CMU ARCTIC corpus), not an absence of accent.


## 1. Setup — Install Dependencies

In [1]:
# Install all required libraries for audio processing, deep learning, and dataset access
!pip install -q librosa numpy soundfile scikit-learn torch torchcodec scipy matplotlib datasets huggingface_hub tqdm nbformat


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Core imports used throughout the notebook
import os
import io
import re
import pickle
import numpy as np
import librosa
import soundfile as sf
import torch
import torch.nn as nn
from tqdm import tqdm

# Create folders to store processed data and outputs
os.makedirs('data/processed', exist_ok=True)

## 2. Feature Extractor

This class extracts the three key acoustic features we need from raw speech audio:
- **MFCCs** (Mel-Frequency Cepstral Coefficients): capture the timbral/phonetic content of speech, and are the main signal we use to model accent differences
- **Pitch (F0)**: the fundamental frequency contour, related to intonation
- **Formants**: resonant frequencies of the vocal tract, strongly correlated with vowel/accent identity


In [3]:
class FeatureExtractor:
    """Extracts MFCC, pitch, and formant features from an audio waveform."""

    def __init__(self, sample_rate=16000, n_mfcc=13, n_fft=1024, hop_length=256):
        # Standard speech-processing settings: 16kHz sample rate matches our datasets
        self.sr = sample_rate
        self.n_mfcc = n_mfcc          # number of MFCC coefficients to extract
        self.n_fft = n_fft            # FFT window size for spectral analysis
        self.hop_length = hop_length  # step size between frames

    def extract_mfcc(self, audio):
        """Extract MFCC features - the primary features used for accent conversion."""
        mfcc = librosa.feature.mfcc(
            y=audio, sr=self.sr, n_mfcc=self.n_mfcc,
            n_fft=self.n_fft, hop_length=self.hop_length
        )
        return mfcc  # shape: (n_mfcc, time_frames)

    def extract_pitch(self, audio):
        """Extract F0 (pitch) contour using the pYIN algorithm, robust to noise."""
        f0, voiced_flag, voiced_probs = librosa.pyin(
            audio, fmin=librosa.note_to_hz('C2'),
            fmax=librosa.note_to_hz('C7'),
            sr=self.sr, hop_length=self.hop_length
        )
        # Unvoiced frames return NaN - replace with 0 for numerical stability
        f0 = np.nan_to_num(f0)
        return f0

    def extract_formants(self, audio, n_formants=3, order=12):
        """
        Estimate formants (F1, F2, F3) per frame using Linear Predictive Coding (LPC).
        Formants carry most of the accent-distinguishing information since they
        reflect vocal tract shape during vowel production.
        """
        frame_length = self.n_fft
        hop = self.hop_length
        formants_over_time = []

        # Slide a window across the audio, analyzing each frame separately
        for start in range(0, len(audio) - frame_length, hop):
            frame = audio[start:start + frame_length] * np.hamming(frame_length)
            try:
                # Fit an LPC model to the frame - its roots correspond to resonances
                lpc_coeffs = librosa.lpc(frame, order=order)
                roots = np.roots(lpc_coeffs)
                roots = roots[np.imag(roots) >= 0]  # keep only upper half of unit circle
                angles = np.arctan2(np.imag(roots), np.real(roots))
                freqs = angles * (self.sr / (2 * np.pi))  # convert angle to frequency
                freqs = sorted(freqs[freqs > 90])  # discard near-DC values
                formants_over_time.append(freqs[:n_formants])
            except Exception:
                # LPC can fail on silent/degenerate frames - fall back to zeros
                formants_over_time.append([0] * n_formants)

        return formants_over_time

    def extract_all(self, audio):
        """Convenience method to extract all three feature types at once."""
        return {
            'mfcc': self.extract_mfcc(audio),
            'pitch': self.extract_pitch(audio),
            'formants': self.extract_formants(audio),
        }

# Quick smoke test to confirm the extractor works before using it on real data
extractor = FeatureExtractor()
_dummy_audio = np.random.randn(16000 * 2).astype(np.float32)  # 2 seconds of noise
_test_features = extractor.extract_all(_dummy_audio)
print("MFCC shape:", _test_features['mfcc'].shape)
print("Pitch shape:", _test_features['pitch'].shape)
print("Number of formant frames:", len(_test_features['formants']))

MFCC shape: (13, 126)
Pitch shape: (126,)
Number of formant frames: 121


## 3. Model Architecture

A lightweight BiLSTM (Bidirectional LSTM) encoder-decoder that learns to map accented MFCC frames toward native-accent MFCC frames. A residual connection helps training stability by having the model predict a *correction* rather than the full output from scratch.


In [4]:
class AccentNeutralizer(nn.Module):
    """BiLSTM-based sequence model for MFCC-level accent conversion."""

    def __init__(self, n_mfcc=13, hidden_dim=128, num_layers=2):
        super().__init__()
        self.n_mfcc = n_mfcc

        # Bidirectional LSTM reads the sequence forward and backward,
        # capturing context from both past and future frames
        self.encoder = nn.LSTM(
            input_size=n_mfcc,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.2 if num_layers > 1 else 0.0
        )

        # Decoder maps the encoded representation back into MFCC space
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),  # *2 because bidirectional
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, n_mfcc)
        )

    def forward(self, x):
        # x shape: (batch, time_frames, n_mfcc)
        encoded, _ = self.encoder(x)
        correction = self.decoder(encoded)
        # Residual connection: predict the delta from input rather than the raw output.
        # This stabilizes training since the output starts close to the input.
        return x + correction

# Smoke test to confirm the architecture works end-to-end
model = AccentNeutralizer()
_dummy_input = torch.randn(4, 100, 13)  # batch=4, time=100 frames, 13 MFCCs
_output = model(_dummy_input)
print("Input shape:", _dummy_input.shape)
print("Output shape:", _output.shape)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Input shape: torch.Size([4, 100, 13])
Output shape: torch.Size([4, 100, 13])
Total parameters: 576,269


## 4. Data Preparation

Builds the training dataset from two public speech corpora:
- **[L2-ARCTIC](https://huggingface.co/datasets/KoelLabs/L2Arctic)**: non-native English speech from speakers of Hindi, Chinese, Spanish, Arabic, Korean, and Vietnamese, reading standardized prompt sentences
- **[CMU ARCTIC](https://huggingface.co/datasets/MikhailT/cmu-arctic)**: native English reference speech, using the *same underlying prompt sentences* as L2-ARCTIC

Since both corpora share the same sentence set, we can match recordings by sentence text to build parallel accented/native pairs. Because accented and native speakers read at different speeds, we then use **Dynamic Time Warping (DTW)** to time-align each pair before training, ensuring the model learns a frame-correspondent mapping rather than a naive position-wise one.

**Caching note:** feature extraction across ~1,148 pairs takes significant time. This notebook checks for previously cached results and loads them instantly if available, only recomputing from scratch if needed.


In [5]:
def normalize_text(text):
    """Lowercase and strip punctuation so sentence text matches across both datasets."""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

CACHE_PATH = 'data/processed/features_final.pkl'
TARGET_LANGUAGES = ['Hindi', 'Chinese']   # accent pairs used in this v1
NATIVE_SPEAKERS = ['bdl', 'slt']          # male + female native reference speakers

if os.path.exists(CACHE_PATH):
    # Fast path: load previously extracted features instead of recomputing
    print("Found cached features, loading...")
    with open(CACHE_PATH, 'rb') as f:
        processed_data = pickle.load(f)
    print(f"Loaded {len(processed_data)} cached feature pairs.")
else:
    # Slow path: download datasets, match sentence pairs, and extract features
    from datasets import load_dataset, Audio

    print("No cache found. Loading datasets from Hugging Face...")
    l2 = load_dataset("KoelLabs/L2Arctic")
    l2 = l2.cast_column("audio", Audio(decode=False))
    cmu = load_dataset("MikhailT/cmu-arctic")
    cmu = cmu.cast_column("audio", Audio(decode=False))

    # Build a lookup of native sentences -> audio, for fast matching
    print("Building native sentence lookup...")
    native_lookup = {}
    for spk in NATIVE_SPEAKERS:
        for row in cmu[spk]:
            key = normalize_text(row['text'])
            if key not in native_lookup:
                native_lookup[key] = row

    # Match each accented sentence to its native counterpart by text
    print("Matching accented/native sentence pairs...")
    pairs = []
    for row in l2['scripted']:
        if row['speaker_native_language'] not in TARGET_LANGUAGES:
            continue
        key = normalize_text(row['text'])
        if key in native_lookup:
            pairs.append({'accented': row, 'native': native_lookup[key], 'language': row['speaker_native_language']})

    print(f"Total matched pairs: {len(pairs)}")

    # Extract MFCC + pitch features for every pair
    processed_data = []
    for i, pair in enumerate(tqdm(pairs, desc="Extracting features")):
        try:
            acc_audio, _ = sf.read(io.BytesIO(pair['accented']['audio']['bytes']))
            nat_audio, _ = sf.read(io.BytesIO(pair['native']['audio']['bytes']))
            acc_feat = extractor.extract_all(acc_audio.astype(np.float32))
            nat_feat = extractor.extract_all(nat_audio.astype(np.float32))
            processed_data.append({
                'text': pair['accented']['text'],
                'language': pair['language'],
                'accented_mfcc': acc_feat['mfcc'],
                'accented_pitch': acc_feat['pitch'],
                'native_mfcc': nat_feat['mfcc'],
                'native_pitch': nat_feat['pitch'],
            })
        except Exception as e:
            continue  # skip any corrupted/unreadable samples

    # Cache the results so future runs (including this notebook's re-runs) are instant
    with open(CACHE_PATH, 'wb') as f:
        pickle.dump(processed_data, f)
    print(f"Extracted and cached {len(processed_data)} pairs.")

Found cached features, loading...
Loaded 1148 cached feature pairs.


In [6]:
def align_pair(acc_mfcc, nat_mfcc):
    """
    Time-align an accented MFCC sequence to a native MFCC sequence using DTW,
    since the two recordings differ in speaking rate/duration.
    """
    D, wp = librosa.sequence.dtw(X=acc_mfcc, Y=nat_mfcc, metric='euclidean')
    wp = wp[::-1]  # warping path comes out reversed; flip to chronological order

    # Average all accented frames that map to each native time-step
    aligned_acc = np.zeros_like(nat_mfcc)
    counts = np.zeros(nat_mfcc.shape[1])
    for acc_idx, nat_idx in wp:
        aligned_acc[:, nat_idx] += acc_mfcc[:, acc_idx]
        counts[nat_idx] += 1
    counts[counts == 0] = 1  # avoid divide-by-zero
    return aligned_acc / counts

def pad_or_crop(mfcc, max_frames):
    """Standardize sequence length by padding with zeros or cropping."""
    n_frames = mfcc.shape[1]
    if n_frames >= max_frames:
        return mfcc[:, :max_frames]
    return np.pad(mfcc, ((0, 0), (0, max_frames - n_frames)), mode='constant')

MAX_FRAMES = 300
X_PATH = 'data/processed/X_train.npy'
Y_PATH = 'data/processed/Y_train.npy'

if os.path.exists(X_PATH) and os.path.exists(Y_PATH):
    # Fast path: reuse previously aligned/padded training tensors
    print("Found cached training tensors, loading...")
    X = np.load(X_PATH)
    Y = np.load(Y_PATH)
else:
    # Slow path: align every pair with DTW and build training tensors
    print("Aligning pairs with DTW...")
    X_list, Y_list = [], []
    for item in tqdm(processed_data, desc="Aligning"):
        try:
            aligned_acc = align_pair(item['accented_mfcc'], item['native_mfcc'])
            X_list.append(pad_or_crop(aligned_acc, MAX_FRAMES).T)
            Y_list.append(pad_or_crop(item['native_mfcc'], MAX_FRAMES).T)
        except Exception:
            continue

    X = np.stack(X_list)
    Y = np.stack(Y_list)
    np.save(X_PATH, X)
    np.save(Y_PATH, Y)

print(f"Training data ready: X={X.shape}, Y={Y.shape}")

Found cached training tensors, loading...
Training data ready: X=(1148, 300, 13), Y=(1148, 300, 13)


## 5. Training

Trains the `AccentNeutralizer` model on the aligned MFCC pairs. If a trained checkpoint already exists, it's loaded instead of retraining from scratch.

In [7]:
from torch.utils.data import Dataset, DataLoader, random_split

class AccentDataset(Dataset):
    """Simple wrapper to feed (accented, native) MFCC pairs into PyTorch's DataLoader."""
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

CHECKPOINT_PATH = 'best_model.pt'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if os.path.exists(CHECKPOINT_PATH):
    # Fast path: load the already-trained model instead of retraining
    print("Found existing trained model, loading checkpoint...")
    model = AccentNeutralizer().to(device)
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
    model.eval()
    print("Model loaded from checkpoint.")
else:
    # Slow path: train the model from scratch
    dataset = AccentDataset(X, Y)
    val_size = int(0.15 * len(dataset))
    train_size = len(dataset) - val_size
    train_ds, val_ds = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

    model = AccentNeutralizer().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()

    EPOCHS = 30
    best_val_loss = float('inf')

    for epoch in range(EPOCHS):
        # --- Training pass ---
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * xb.size(0)
        train_loss /= train_size

        # --- Validation pass (no gradient updates) ---
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb)
                val_loss += criterion(pred, yb).item() * xb.size(0)
        val_loss /= val_size

        print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f}")

        # Save the model whenever validation loss improves
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), CHECKPOINT_PATH)

    print(f"Training complete. Best val loss: {best_val_loss:.4f}")

Using device: cpu
Found existing trained model, loading checkpoint...
Model loaded from checkpoint.


## 6. Inference — Try It Out

Runs the full pipeline end-to-end: takes an accented audio clip, converts its MFCCs toward the native accent, and reconstructs audible output using Griffin-Lim phase estimation.

In [8]:
def neutralize_accent(audio, model, extractor, max_frames=MAX_FRAMES, sr=16000):
    """Convert a single accented audio waveform toward the native-accent target."""
    mfcc = extractor.extract_mfcc(audio)
    T = mfcc.shape[1]

    # Pad or crop to match the fixed length the model was trained on
    mfcc_input = pad_or_crop(mfcc, max_frames)

    # Run through the trained model
    x = torch.tensor(mfcc_input.T, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(x)
    predicted_mfcc = output.squeeze(0).cpu().numpy().T
    predicted_mfcc = predicted_mfcc[:, :T]  # trim back to original length

    # Reconstruct an audible waveform from the converted MFCCs
    converted_audio = librosa.feature.inverse.mfcc_to_audio(
        predicted_mfcc, sr=sr, n_fft=1024, hop_length=256
    )
    return converted_audio

# Demo: run on a sample Hindi-accented sentence from L2-ARCTIC
from datasets import load_dataset, Audio

print("Loading a sample accented sentence for demo...")
l2_demo = load_dataset("KoelLabs/L2Arctic")
l2_demo = l2_demo.cast_column("audio", Audio(decode=False))

demo_sample = None
for row in l2_demo['scripted']:
    if row['speaker_native_language'] == 'Hindi':
        demo_sample = row
        break

print(f"Demo sentence: {demo_sample['text']}")
demo_audio, _ = sf.read(io.BytesIO(demo_sample['audio']['bytes']))
demo_audio = demo_audio.astype(np.float32)

converted = neutralize_accent(demo_audio, model, extractor)

sf.write('demo_original.wav', demo_audio, 16000)
sf.write('demo_converted.wav', converted, 16000)
print("Saved demo_original.wav and demo_converted.wav for comparison.")

C:\Users\bharg\Documents\gssoc-projects\ML-CaPsule\Accent_Neutralizer_for_Speech\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using the latest cached version of the dataset since KoelLabs/L2Arctic couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\bharg\.cache\huggingface\datasets\KoelLabs___l2_arctic\default\0.0.0\e9f2aaca153989a45e86bc2d981ba03c4879476d (last modified on Fri Jul 10 17:04:35 2026).


Loading a sample accented sentence for demo...
Demo sentence: for the twentieth time that evening the two men shook hands
Saved demo_original.wav and demo_converted.wav for comparison.


## Results (v1)

- Training loss decreased consistently over 30 epochs (621 → 350 train, 597 → 384 validation), with train/val loss tracking closely — indicating real learning without significant overfitting.
- Qualitatively, converted audio remains intelligible (sentence content is preserved) with an audible timbral shift from the source accent.

## Known Limitations

- **Audio quality**: Griffin-Lim reconstruction introduces a robotic/synthetic quality, since it estimates phase information that was discarded during MFCC extraction. This is a reconstruction-method limitation, not a failure of the conversion model itself.
- **Speaker identity preservation**: this v1 does not explicitly disentangle speaker identity from accent — some voice-identity drift is expected.
- **Prosody**: pitch/rhythm patterns are extracted but not yet incorporated into the conversion model itself.
- **Accent coverage**: limited to Hindi and Chinese non-native accents in v1.

## Future Work

- Replace Griffin-Lim with a neural vocoder (e.g., HiFi-GAN) for natural-sounding output
- Incorporate prosody (pitch/rhythm) into the conversion model
- Explicit speaker-identity preservation via separate content/speaker embeddings
- Extend to additional accent pairs (Spanish, Arabic, Korean, Vietnamese)

## Acknowledgements

- [L2-ARCTIC corpus](https://psi.engr.tamu.edu/l2-arctic-corpus/) (Texas A&M University, Iowa State University)
- [CMU ARCTIC corpus](http://festvox.org/cmu_arctic/) (Carnegie Mellon University)
